# 07 — Automating the Judges, and Finding Out If You Can Trust Them

Notebooks 05 and 06 were you, reading 20 answers, writing down a faithfulness verdict and a correctness verdict for each, with reasoning. That was the expensive, slow, completely trustworthy version. This notebook builds the fast version -- an LLM call that does the same two jobs -- and then does the thing most learning plans skip entirely: checks whether the fast version actually agrees with you, on your own data, before trusting it on anything bigger.


In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"

DATA_DIR = os.path.join("..", "data")
with open(os.path.join(DATA_DIR, "eval_examples.json")) as f:
    examples = json.load(f)


## Building the faithfulness judge

This is not narrating your evaluation to you -- it's the same mechanical task you did by hand in notebook 05, in code: extract claims, check each against the context, return structured output.


In [ ]:
FAITHFULNESS_PROMPT = """You are a faithfulness judge. Extract every factual claim in the \
answer below, then mark each as "supported" or "unsupported" based only on the context. \
Return strict JSON only, no other text, in this shape:
{{"claims": [{{"text": "...", "supported": true, "reason": "..."}}]}}

Context:
{context}

Answer:
{answer}"""


def judge_faithfulness(context: str, answer: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": FAITHFULNESS_PROMPT.format(context=context, answer=answer)}],
        temperature=0.0,
        reasoning_effort="low",
    )
    return json.loads(response.choices[0].message.content)


## Testing it on the two trickiest cases from notebook 05

Not the easy ones -- the inversion (example 17) and the scope-shift (example 19), the two failures worked example 1 and 2 specifically called out as easy to miss.


In [ ]:
result_17 = judge_faithfulness(examples[17]["context_text"], examples[17]["system_answer_overconfident"])
print("Example 17 (inversion):")
for c in result_17["claims"]:
    print(f"  {c['supported']}: {c['text']}")
    print(f"    -> {c['reason']}")

print()
result_19 = judge_faithfulness(examples[19]["context_text"], examples[19]["system_answer"])
print("Example 19 (scope-shift):")
for c in result_19["claims"]:
    print(f"  {c['supported']}: {c['text']}")
    print(f"    -> {c['reason']}")


If the judge caught both -- flagged the inversion as unsupported *and* explained that the context says "medieval," not "modern" -- that's a genuinely good sign. Those two failures were specifically chosen because a shallow "do these words appear in the context" check would miss them. A judge that catches them is doing something closer to real reading comprehension than keyword matching.


## Building the correctness judge

Same idea, second metric: does the answer match gold in meaning, allowing for the partially-correct middle ground notebook 06 introduced.


In [ ]:
CORRECTNESS_PROMPT = """Expected answer: {gold}
Actual answer: {answer}

Is the actual answer correct and equivalent to the expected answer? Consider partial matches -- \
an answer that supports a real but different clause than the expected one should be scored \
PARTIALLY_CORRECT, not INCORRECT. If the expected answer is "Not stated in the documents" and \
the actual answer honestly declines to answer, that counts as CORRECT.

Return strict JSON only: {{"verdict": "CORRECT|PARTIALLY_CORRECT|INCORRECT", "reason": "..."}}"""


def judge_correctness(gold: str, answer: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": CORRECTNESS_PROMPT.format(gold=gold, answer=answer)}],
        temperature=0.0,
        reasoning_effort="low",
    )
    return json.loads(response.choices[0].message.content)


## Running both judges on all 20, then comparing to your own notes

This is the actual calibration step. Not "does the framework seem reasonable" -- does it agree with the specific judgment calls you already made, with reasoning, in notebooks 05 and 06.


In [ ]:
auto_faithfulness = {}
auto_correctness = {}

for i, ex in enumerate(examples):
    auto_faithfulness[i] = judge_faithfulness(ex["context_text"], ex["system_answer"])
    auto_correctness[i] = judge_correctness(ex["gold_answer"], ex["system_answer"])

print("Automated judging complete for all 20 examples.")


In [ ]:
# Your manual verdicts from notebooks 05 and 06, carried over for direct comparison
manual_correctness = {
    0: "fully_correct",
    7: "partially_correct",
    13: "incorrect_vs_gold_but_gold_is_questionable",
    16: "fully_correct",
}

print(f"{'#':<4}{'Your verdict':<45}{'Judge verdict':<20}{'Agree?'}")
for i, manual in manual_correctness.items():
    auto = auto_correctness[i]["verdict"]
    manual_normalized = "CORRECT" if "correct" in manual and "partially" not in manual and "incorrect" not in manual else (
        "PARTIALLY_CORRECT" if "partially" in manual else "INCORRECT"
    )
    agree = "yes" if manual_normalized == auto else "NO -- investigate"
    print(f"{i:<4}{manual:<45}{auto:<20}{agree}")


## Where they actually disagreed

If example `7` came back `INCORRECT` from the automated judge, that's a real, useful disagreement, not a bug to fix. You called it partially correct because the system found a real, different liability clause in the same contract. The judge prompt explicitly says to score exactly that case as `PARTIALLY_CORRECT` -- and it may still have called it `INCORRECT` anyway. That tells you something specific: this judge, on this model, defaults toward strict gold-matching even when explicitly instructed to give partial credit for a defensible near-miss. That's a real calibration finding about *this* judge on *your* data -- not a general fact about all LLM judges everywhere.

And example `13` is the sharper limitation. You know, from notebook 02's investigation, that the gold label itself is questionable -- both buildings really are real estate. The automated judge has no way to know that. It only ever compares against the label it's given, and the label here doesn't hold up under the same scrutiny you'd apply to any system output. **An LLM judge is exactly as good as the ground truth you hand it, and no better.** This is the single most important limitation to carry into notebook `09`, when frameworks start making these same comparisons look effortless.


## Is this judge trustworthy, or only a first pass?

Answer honestly, based on what you just saw, not in the abstract:

- Did it catch the subtle failures (inversion, scope-shift) that a keyword-overlap check would have missed? If yes, that's a real point in its favor.
- Did it disagree with you anywhere it shouldn't have (like example 7, if that's what happened)? Note the direction of the disagreement -- stricter than you, or more lenient?
- Did it have any way to catch example 13's questionable gold label? It didn't, and it can't, by construction -- no LLM judge can independently verify a benchmark's own ground truth.

None of this makes the judge useless. It makes it a fast first pass that still needs your own spot-checking on anything that matters -- which is a very different, much more honest claim than "the judge is right."


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| LLM-as-judge | Using a model to automate a judgment a human would otherwise make by hand |
| Calibration | Checking an automated judge's verdicts against your own, on the same data, before trusting it further |
| Self-preference bias | A judge's tendency to favor answers stylistically similar to its own outputs -- one reason calibration matters even when a judge model seems capable |

The realization this notebook is built around: a framework isn't automating "correctness" in some abstract sense. It's automating *your* judgment, or it's automating nobody's -- and the only way to know which is to actually compare it against a judgment you trust, on data you understand, before it runs unsupervised on 500 examples you don't have time to read.

**Next up:** notebook `08` packages `judge_faithfulness` and `judge_correctness` into reusable classes -- the moment where "Framework != Evaluation, Framework = Automation" stops being an abstract slogan and becomes something you can point at in your own code.

**Exercises:**

- Run the full comparison table (not just the 4 indices above) against every verdict you wrote in notebooks 05 and 06.
- For any disagreement you find, write one sentence on whether the judge was too strict, too lenient, or just differently reasoned -- not just "wrong."
- Try raising the judge's `temperature` from 0.0 to 0.5 and re-run it on example 7 a few times. Does the verdict stay consistent, or does it start flipping between runs?
